# Step 2 · Preview Jev's typed questions

Preview Jev's typed yes/no pathogen and pair-feature questions with the selected pathogen, host, title, and abstract in state. The notebook makes no model or network calls. Jev returns probabilities, not generated JSON, rationale, or evidence text.


## Step 1 · Load the run and prompt/profile definitions

In [1]:
import json, sys
from pathlib import Path
import pandas as pd
import yaml
from IPython.display import display, Markdown

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'pathogen_atlas_v4').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'pathogen_atlas_v4').is_dir():
    raise RuntimeError('Open this notebook from the V4 repository.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / 'notebooks'))
from _shared import configure_notebook, configured_run_id
from pathogen_atlas_v4.features import resolve_profile
from pathogen_atlas_v4.prompts import build_pathogen_screen_question, build_pair_feature_question
from pathogen_atlas_v4.runs import resolve_run

ROOT = PROJECT_ROOT
CONFIG = yaml.safe_load((ROOT / 'configs/pilot_v1.yaml').read_text(encoding='utf-8'))
RUN_ID = configured_run_id(ROOT)
ROOT, LOG, LOG_PATH = configure_notebook('02_preview_prompts', RUN_ID)
TAXONOMY_ROOT = sorted((ROOT / 'data/v4/staging').glob('*/taxonomy'))[-1]
RUN_TAXONOMY_ROOT = ROOT / 'data/v4/runs' / RUN_ID / 'taxonomy'
if not (RUN_TAXONOMY_ROOT / 'pathogens.parquet').exists():
    raise FileNotFoundError('Run taxonomy was not resolved; run 00_configure_run.ipynb first.')
PATHOGENS = pd.read_parquet(RUN_TAXONOMY_ROOT / 'pathogens.parquet')
HOSTS = pd.read_parquet(TAXONOMY_ROOT / 'hosts.parquet')
CATALOGUE = yaml.safe_load((ROOT / CONFIG['features']['catalogue']).read_text(encoding='utf-8'))
PROMPT_PATH = ROOT / CONFIG.get('decisions', {}).get('prompt_catalogue', 'configs/prompts.yaml')
PROMPTS = yaml.safe_load(PROMPT_PATH.read_text(encoding='utf-8'))
print('Run:', RUN_ID, '| taxonomy:', TAXONOMY_ROOT.name, '| prompt set:', PROMPTS['prompt_set_version'])
print('Log:', LOG_PATH)
LOG.info('Loaded prompt preview inputs: run=%s taxonomy=%s prompt_set=%s', RUN_ID, TAXONOMY_ROOT.name, PROMPTS['prompt_set_version'])

2026-11-02 21:44 INFO Notebook session started at 2026-11-02T21:44:33.861368+00:00


2026-11-02 21:44 INFO Project root: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4


2026-11-02 21:44 INFO Run log: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/data/v4/logs/pilot-v1/02_preview_prompts.log


2026-11-02 21:44 INFO Loaded prompt preview inputs: run=pilot-v1 taxonomy=taxonomy prompt_set=system1-prompt-set-v2


Run: pilot-v1 | taxonomy: taxonomy | prompt set: system1-prompt-set-v2
Log: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/data/v4/logs/pilot-v1/02_preview_prompts.log


## Step 2 · Resolve one pathogen–host pair to preview

In [2]:
# Leave either ID as None to use the first resolved selection. Set an explicit ID to inspect a different member of the run.
PREVIEW_PATHOGEN_ID = None
PREVIEW_HOST_ID = None
PREVIEW_ARTICLE_TITLE = ''
PREVIEW_ARTICLE_ABSTRACT = ''
PREVIEW_FEATURE_ID = None  # None selects the first applicable pathogen_host_pair feature.

host_groups = {key: group.host_id.astype(str).tolist() for key, group in HOSTS.groupby('host_group')}
pathogen_groups = {key: group.pathogen_id.astype(str).tolist() for key, group in PATHOGENS.groupby('pathogen_group')}
resolved = resolve_run(CONFIG, known_host_ids=set(HOSTS.host_id.astype(str)),
    known_pathogen_ids=set(PATHOGENS.pathogen_id.astype(str)),
    host_group_ids=host_groups, pathogen_group_ids=pathogen_groups)
if not resolved.pathogen_ids or not resolved.host_ids:
    raise ValueError('The run must resolve at least one pathogen and one host before prompt preview.')
pathogen_id = str(PREVIEW_PATHOGEN_ID or resolved.pathogen_ids[0])
host_id = str(PREVIEW_HOST_ID or resolved.host_ids[0])
if pathogen_id not in set(resolved.pathogen_ids):
    raise ValueError(f'{pathogen_id} is not in the resolved run pathogen selection.')
if host_id not in set(resolved.host_ids):
    raise ValueError(f'{host_id} is not in the resolved run host selection.')
pathogen = PATHOGENS.set_index('pathogen_id').loc[pathogen_id]
host = HOSTS.set_index('host_id').loc[host_id]
print(f"Pathogen: {pathogen.accepted_name} [{pathogen.pathogen_group}] ({pathogen_id})")
print(f"Host: {host.accepted_name} [{host.host_group}] ({host_id})")
LOG.info('Prompt preview pair selected: pathogen=%s host=%s', pathogen_id, host_id)

2026-11-02 21:44 INFO Prompt preview pair selected: pathogen=viruses:868bbe896ed10f2211ce host=ncbi:4530


Pathogen: Aalivirus apekidu [viruses] (viruses:868bbe896ed10f2211ce)
Host: Oryza sativa [crop] (ncbi:4530)


## Step 3 · Print the pathogen Jev question preview

In [3]:
import textwrap


def print_prompt_readable(prompt, width=80):
    """Print prompt fields with long text wrapped for notebook readability."""
    for key, value in prompt.items():
        if isinstance(value, str):
            print(f"{key}:")
            for paragraph in value.splitlines():
                if paragraph:
                    print(textwrap.fill(
                        paragraph, width=width,
                        initial_indent='  ', subsequent_indent='  '))
                else:
                    print()
        elif isinstance(value, (dict, list)):
            print(f"{key}:")
            rendered = json.dumps(value, indent=2, ensure_ascii=False)
            print(textwrap.indent(rendered, '  '))
        else:
            print(f"{key}: {value}")


pathogen_question = build_pathogen_screen_question(
    pathogen_name=str(pathogen.accepted_name), pathogen_id=pathogen_id,
    host_name=str(host.accepted_name), host_id=host_id,
    article_title=PREVIEW_ARTICLE_TITLE,
    article_abstract=PREVIEW_ARTICLE_ABSTRACT,
    prompt_config=PROMPTS)
print_prompt_readable(pathogen_question)
display(Markdown(f"**Question version:** `{pathogen_question['question_version']}` · **Jev question version:** `{pathogen_question['prompt_version']}`"))
LOG.info('Printed pathogen screening question question_version=%s jev_question=%s', pathogen_question['question_version'], pathogen_question['prompt_version'])


question_version:
  pathogen-screen-system1-v2-threshold-0.7
prompt_version:
  pathogen-screen-noul-v1
state:
  {
    "pathogen": {
      "name": "Aalivirus apekidu",
      "id": "viruses:868bbe896ed10f2211ce"
    },
    "host": {
      "name": "Oryza sativa",
      "id": "ncbi:4530"
    },
    "article": {
      "title": "",
      "abstract": ""
    }
  }
questions:
  {
    "pathogen_screen": {
      "type": "noul",
      "instructions": "Does `article.abstract` directly establish that `pathogen.name` causes disease in `host.name`? Assess only this abstract and the named pathogen–host pair. Do not count a title-only match, a mere mention, colonization or carriage without disease, a vector or reservoir role, experimental use, or food contamination alone. Do not infer beyond the text.",
      "criteria": {
        "true": "The abstract directly supports disease caused by this named organism in this selected host.",
        "false": "The abstract does not directly support that exact path

**Question version:** `pathogen-screen-system1-v2-threshold-0.7` · **Jev question version:** `pathogen-screen-noul-v1`

2026-11-02 21:44 INFO Printed pathogen screening question question_version=pathogen-screen-system1-v2-threshold-0.7 jev_question=pathogen-screen-noul-v1


## Step 4 · Print every feature selected for this pathogen–host pair

The resolved profile includes every applicable feature from the core, pathogen-group, host-group, host-taxon, and pair-context modules. Review this full list before inspecting the example pair feature prompt.


In [4]:
profile = resolve_profile(CATALOGUE,
    pathogen_group=str(pathogen.pathogen_group), host_group=str(host.host_group),
    host_taxon_id=str(host.taxon_id), relationship_context='disease',
    profile_version=str(CONFIG['features']['profile_version']))

all_features = profile['ordered_features']
feature_table = pd.DataFrame([{
    'order': index,
    'module': item.get('module', ''),
    'scope': item.get('scope', ''),
    'feature_id': item.get('feature_id', ''),
    'question_version': item.get('question_version', ''),
    'question': item.get('question', ''),
    'review_status': item.get('review_status', ''),
} for index, item in enumerate(all_features, start=1)])
print('Resolved profile:', profile['profile_id'], profile['profile_version'], profile['resolved_profile_hash'])
print(f'All selected features for {pathogen.accepted_name} × {host.accepted_name}: {len(feature_table)}')
display(feature_table)
LOG.info('Printed all selected pair profile features: count=%d profile=%s', len(feature_table), profile['resolved_profile_hash'])

pair_features = [item for item in all_features if item.get('scope') == 'pathogen_host_pair']
if PREVIEW_FEATURE_ID:
    pair_features = [item for item in pair_features if item['feature_id'] == PREVIEW_FEATURE_ID]
if not pair_features:
    print('This profile has no selected pathogen_host_pair feature to preview.')
else:
    feature = pair_features[0]
    feature_question = build_pair_feature_question(
        pathogen_name=str(pathogen.accepted_name), pathogen_id=pathogen_id,
        host_name=str(host.accepted_name), host_id=host_id, feature=feature,
        article_title=PREVIEW_ARTICLE_TITLE, article_abstract=PREVIEW_ARTICLE_ABSTRACT,
        prompt_config=PROMPTS)
    print('Example pair feature:', feature['feature_id'], '| question version:', feature.get('question_version', ''))
    print_prompt_readable(feature_question)
    LOG.info('Printed pair feature question: feature=%s question=%s jev_question=%s profile=%s',
             feature['feature_id'], feature.get('question_version'), feature_question['prompt_version'], profile['resolved_profile_hash'])


Resolved profile: host-aware-default host-aware-profile-v1 0c77347f7d67a8f38fef9d43299bd7a6d6c28b6ad732aa584ae9ade4b3b8e830
All selected features for Aalivirus apekidu × Oryza sativa: 1


,order,module,scope,feature_id,question_version,question,review_status
0,1,core,pathogen_host_pair,abstract_reports_disease,disease-report-v1,Does this abstract report disease in the speci...,pilot


2026-11-02 21:44 INFO Printed all selected pair profile features: count=1 profile=0c77347f7d67a8f38fef9d43299bd7a6d6c28b6ad732aa584ae9ade4b3b8e830


2026-11-02 21:44 INFO Printed pair feature question: feature=abstract_reports_disease question=disease-report-v1 jev_question=pair-feature-noul-v1 profile=0c77347f7d67a8f38fef9d43299bd7a6d6c28b6ad732aa584ae9ade4b3b8e830


Example pair feature: abstract_reports_disease | question version: disease-report-v1
feature_id:
  abstract_reports_disease
question_version:
  disease-report-v1
prompt_version:
  pair-feature-noul-v1
state:
  {
    "feature_id": "abstract_reports_disease",
    "feature_question_version": "disease-report-v1",
    "feature_question": "Does this abstract report disease in the specified host caused by the specified pathogen?",
    "pathogen": {
      "name": "Aalivirus apekidu",
      "id": "viruses:868bbe896ed10f2211ce"
    },
    "host": {
      "name": "Oryza sativa",
      "id": "ncbi:4530"
    },
    "relationship_context": "disease",
    "article": {
      "title": "",
      "abstract": ""
    }
  }
questions:
  {
    "pair_feature": {
      "type": "noul",
      "instructions": "Does `article.title` or `article.abstract` support a yes answer to `feature_question` for `pathogen.name` in `host.name`? Assess only the supplied article and the stated pair. Do not infer beyond the text."

## Review notes

The table lists every feature selected for the chosen pathogen–host pair, including each feature's module, scope, question, and version. The printed example is the first selected pair-scoped feature. To preview against a real article, enter its PubMed title and abstract in Step 2 and rerun Steps 2–4. No decision is made or cached by this preview.
